# Moody Player: FER-2013 emotion classifier

This notebook trains, evaluates and exports the emotion classifier that runs in the Moody Player web app.

It compares two models on FER-2013 (48×48 grayscale faces, 7 emotions):

| | Model | Idea |
|---|---|---|
| (a) | **Small CNN from scratch** | 4 conv blocks, about 1.2M parameters, sized for the browser |
| (b) | **Transfer learning** | an ImageNet-pretrained ResNet18 backbone, fine-tuned on FER-2013 |

The pipeline: load data → stratified train/val split → augmentation → weighted loss → early stopping on validation macro-F1 → test-set metrics and confusion matrices → latency and size measurement → ONNX export and validation against PyTorch.

**How to run (Colab)**
1. *Runtime → Change runtime type → T4 GPU*.
2. No Kaggle setup is needed: the current Kaggle CLI downloads this public dataset anonymously. If Kaggle ever refuses, the cell asks for your API token (Kaggle → Settings → API, `KGAT_...`). To skip that prompt, add a Colab Secret named `KAGGLE_API_TOKEN` with *Notebook access* on.
3. *Runtime → Run all*. It takes about 30–45 minutes on a T4.
4. At the end, `moody_player_ml_outputs.zip` downloads. Unzip it into `ml/` in the repo.

All numbers this notebook reports are computed from the run itself. Nothing is hardcoded.

In [ ]:
import os
# Must be set before CUDA initialises, for deterministic cuBLAS.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import subprocess, sys
try:
    import onnx, onnxruntime  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript"])

## 1. Configuration

In [ ]:
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"  # local CPU check on fake data; leave unset on Colab

SEED = 42
CLASSES = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]  # FER-2013 label order
IMG_SIZE = 48
VAL_FRACTION = 0.10
BATCH_SIZE = 128
NUM_WORKERS = 2

TRANSFER_BACKBONE = "resnet18"  # alternative: "mobilenet_v3_small"
TRANSFER_INPUT_SIZE = 96        # 48x48 is upsampled inside the model so the ImageNet backbone keeps spatial detail

CNN_CFG = dict(max_epochs=60, patience=10, lr=1e-3, weight_decay=1e-4)
TRANSFER_CFG = dict(max_epochs=30, patience=7, lr_backbone=1e-4, lr_head=1e-3, weight_decay=1e-4)

# Model selection rule, fixed before looking at any test numbers:
# prefer the smaller model unless the larger one beats it by more than this on VALIDATION macro-F1.
SELECTION_TOLERANCE = 0.01

DATA_DIR = "/content/data/fer2013"
OUT_DIR = "/content/outputs"

if SMOKE_TEST:
    _work = os.environ.get("SMOKE_DIR", "smoke")
    DATA_DIR, OUT_DIR = f"{_work}/data", f"{_work}/outputs"
    NUM_WORKERS = 0
    CNN_CFG.update(max_epochs=2, patience=1)
    TRANSFER_CFG.update(max_epochs=2, patience=1)

In [ ]:
import json, random, time, copy, platform, shutil, io
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.transforms import v2 as T
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib
if SMOKE_TEST:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path(OUT_DIR)
FIG = OUT / "figures"
ONNX_DIR = OUT / "onnx"
for d in (OUT, FIG, ONNX_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else platform.processor())

### Reproducibility
Every source of randomness gets the same seed: Python, NumPy, PyTorch CPU/GPU, the DataLoader shuffling and its worker processes. cuDNN autotuning is turned off and deterministic algorithms are requested. A few CUDA kernels, such as the backward pass of adaptive average pooling, have no deterministic version. PyTorch warns about those instead of failing, so GPU reruns can differ in the last decimal places.

In [ ]:
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)


seed_everything()
torch.use_deterministic_algorithms(True, warn_only=True)

## 2. Data: FER-2013
This uses the `msambare/fer2013` Kaggle copy: `train/` (28,709 images) and `test/` (7,178 images, the official PublicTest + PrivateTest sets), one folder per emotion. The **test set is used only once**, at the end. Validation is a stratified 10% taken from `train/`, so each class keeps the same proportion in both parts.

In [ ]:
def make_fake_dataset(root, per_class={"train": 40, "test": 12}):
    # SMOKE_TEST only: random images in the real folder layout.
    rng = np.random.default_rng(SEED)
    for split, n in per_class.items():
        for name in CLASSES:
            d = Path(root) / split / name
            d.mkdir(parents=True, exist_ok=True)
            for i in range(n):
                Image.fromarray(rng.integers(0, 256, (IMG_SIZE, IMG_SIZE), dtype=np.uint8)).save(d / f"{i}.jpg")


def _colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:  # not on Colab, secret missing, or notebook access not granted
        return None


def load_kaggle_secrets():
    # Optional: public datasets download anonymously with the current Kaggle CLI.
    if os.environ.get("KAGGLE_API_TOKEN") or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        return
    token = _colab_secret("KAGGLE_API_TOKEN")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
        return
    user, key = _colab_secret("KAGGLE_USERNAME"), _colab_secret("KAGGLE_KEY")
    if user and key:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key


def ensure_dataset():
    if (Path(DATA_DIR) / "train").exists():
        return
    if SMOKE_TEST:
        make_fake_dataset(DATA_DIR)
        return
    load_kaggle_secrets()
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"])  # new tokens need a recent CLI
    cmd = ["kaggle", "datasets", "download", "-d", "msambare/fer2013", "-p", DATA_DIR, "--unzip"]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        from getpass import getpass
        os.environ["KAGGLE_API_TOKEN"] = getpass("Kaggle refused the download. Paste your Kaggle API token (KGAT_...): ").strip()
        subprocess.check_call(cmd)
    assert (Path(DATA_DIR) / "train").exists(), f"unexpected dataset layout in {DATA_DIR}"


ensure_dataset()

In [ ]:
def load_split(split):
    xs, ys = [], []
    for label, name in enumerate(CLASSES):
        for f in sorted((Path(DATA_DIR) / split / name).glob("*")):
            img = Image.open(f).convert("L")
            if img.size != (IMG_SIZE, IMG_SIZE):
                img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            xs.append(np.asarray(img, dtype=np.uint8))
            ys.append(label)
    return np.stack(xs), np.array(ys, dtype=np.int64)


X_trainval, y_trainval = load_split("train")
X_test, y_test = load_split("test")
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VAL_FRACTION, stratify=y_trainval, random_state=SEED)

split_counts = {
    name: {c: int((y == i).sum()) for i, c in enumerate(CLASSES)}
    for name, y in (("train", y_train), ("val", y_val), ("test", y_test))
}
print(f"{'class':<10}" + "".join(f"{s:>8}" for s in split_counts))
for c in CLASSES:
    print(f"{c:<10}" + "".join(f"{split_counts[s][c]:>8}" for s in split_counts))
print(f"{'total':<10}" + "".join(f"{sum(v.values()):>8}" for v in split_counts.values()))

In [ ]:
# Plot style shared by every figure: light surface, recessive axes, palette slots 1-2 for the two models.
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
MODEL_COLORS = {"small_cnn": "#2a78d6", "transfer": "#eb6834"}
SEQ_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "text.color": INK, "axes.titlecolor": INK, "axes.grid": False, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

fig, axes = plt.subplots(2, 7, figsize=(12, 3.8))
for i, c in enumerate(CLASSES):
    idx = np.where(y_train == i)[0][:2]
    for r, j in enumerate(idx):
        axes[r, i].imshow(X_train[j], cmap="gray", vmin=0, vmax=255)
        axes[r, i].axis("off")
    axes[0, i].set_title(f"{c}\n(n={split_counts['train'][c]})", fontsize=9)
fig.suptitle("FER-2013 training samples per class", y=1.02)
fig.savefig(FIG / "samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Class imbalance and weighted loss
FER-2013 is very imbalanced. *happy* has about 16× more training images than *disgust*. With a plain cross-entropy loss, the model can ignore *disgust* almost entirely and still score well on accuracy.

**Weighted cross-entropy** multiplies each sample's loss by a weight for its class, set to `N / (K · n_c)` (inverse class frequency):
- `N`: number of training images
- `K`: number of classes
- `n_c`: number of images in class `c`

A mistake on a rare class therefore costs as much in total as mistakes on a common one. The trade-off is that overall accuracy usually drops slightly, because the model now predicts rare classes more often, while recall on rare classes and macro-F1 go up. Alternatives would be oversampling (a weighted sampler) or focal loss. Weighted loss is the simplest and keeps every epoch the same size.

In [ ]:
class_counts = np.bincount(y_train, minlength=len(CLASSES))
class_weights = class_counts.sum() / (len(CLASSES) * class_counts)
for c, n, w in zip(CLASSES, class_counts, class_weights):
    print(f"{c:<10} n={n:>6}  weight={w:.3f}")
CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32)

## 4. Preprocessing contract and augmentation
**The browser must feed the model exactly what training fed it.** To keep that simple, the model input is a single-channel `float32` tensor of shape `[1, 1, 48, 48]` with values in `[0, 1]`, which is just grayscale pixels ÷ 255.

Normalization (subtracting the mean and dividing by the standard deviation) happens **inside the model**, as fixed buffers that are exported to ONNX with it. The browser then only has to do three things, and can't get the mean/std wrong:
1. convert the face crop to grayscale (0.299R + 0.587G + 0.114B)
2. resize it to 48×48
3. divide by 255

**Augmentation**, applied to training images only, creates plausible variations, so the model learns emotions instead of memorising pixels:
- random crops with rescaling (scale 0.8–1.0), which simulate a loose or tight face detector box
- horizontal flips, since expressions are roughly symmetric
- ±10° rotations, for head tilt

In [ ]:
TRAIN_TF = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1), antialias=True),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
])


class FERDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = torch.from_numpy(images).unsqueeze(1)  # uint8 [N, 1, 48, 48]
        self.labels = torch.from_numpy(labels)
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        x = self.images[i].float() / 255.0  # the only preprocessing; the browser does the same
        if self.transform:
            x = self.transform(x)
        return x, self.labels[i]


def make_loader(images, labels, train):
    ds = FERDataset(images, labels, TRAIN_TF if train else None)
    g = torch.Generator().manual_seed(SEED)
    return torch.utils.data.DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=train, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        worker_init_fn=seed_worker, generator=g, persistent_workers=NUM_WORKERS > 0)


val_loader = make_loader(X_val, y_val, train=False)
test_loader = make_loader(X_test, y_test, train=False)

DATA_MEAN = float(X_train.mean() / 255.0)
DATA_STD = float(X_train.std() / 255.0)
print(f"train pixel mean={DATA_MEAN:.4f} std={DATA_STD:.4f}")

# Preview: one image, eight random augmentations.
preview = FERDataset(X_train[:1], y_train[:1], TRAIN_TF)
fig, axes = plt.subplots(1, 9, figsize=(12, 1.8))
axes[0].imshow(X_train[0], cmap="gray", vmin=0, vmax=255)
axes[0].set_title("original", fontsize=9)
for ax in axes[1:]:
    ax.imshow(preview[0][0][0], cmap="gray", vmin=0, vmax=1)
for ax in axes:
    ax.axis("off")
fig.savefig(FIG / "augmentation_preview.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Models
**CNN (convolutional neural network).** A convolution slides small learned filters (3×3 here) over the image. Early layers respond to edges and textures, and deeper layers combine those into parts such as a raised eyebrow or an open mouth. Pooling halves the resolution, so later filters "see" a larger area of the face. Batch normalization stabilises training and dropout reduces overfitting. Global average pooling then turns the final feature maps into one vector for the linear classifier.

The small CNN has 4 blocks (32→64→128→256 channels, 48→24→12→6→3 pixels) and about 1.2M parameters, small enough to download and run in a browser.

**Transfer learning.** Rather than learning everything from 26k images, we start from a ResNet18 that already learned general visual features on 1.2M ImageNet photos, and **fine-tune** it on FER-2013. The new 7-class head starts from random weights, so it gets a 10× higher learning rate than the pretrained backbone. A high rate on the backbone would wipe out what it already learned.

The wrapper adapts our 1-channel 48×48 input to what the backbone expects: it upsamples to 96×96, repeats the gray channel 3 times and applies ImageNet normalization. All of this is inside the exported graph, so both models share the same browser input contract.

In [ ]:
class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32).view(1, -1, 1, 1))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32).view(1, -1, 1, 1))

    def forward(self, x):
        return (x - self.mean) / self.std


def conv_block(cin, cout, dropout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.MaxPool2d(2), nn.Dropout(dropout),
    )


class SmallCNN(nn.Module):
    def __init__(self, num_classes, mean, std):
        super().__init__()
        self.norm = Normalize([mean], [std])
        self.features = nn.Sequential(
            conv_block(1, 32, 0.1), conv_block(32, 64, 0.2), conv_block(64, 128, 0.3), conv_block(128, 256, 0.4))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.5), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.head(self.features(self.norm(x)))


class TransferNet(nn.Module):
    def __init__(self, backbone, num_classes, input_size, pretrained=True):
        super().__init__()
        self.input_size = input_size
        self.norm = Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet statistics
        if backbone == "resnet18":
            net = torchvision.models.resnet18(
                weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            net.fc = nn.Linear(net.fc.in_features, num_classes)
            self.head_prefix = "backbone.fc."
        elif backbone == "mobilenet_v3_small":
            net = torchvision.models.mobilenet_v3_small(
                weights=torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None)
            net.classifier[3] = nn.Linear(net.classifier[3].in_features, num_classes)
            self.head_prefix = "backbone.classifier."
        else:
            raise ValueError(backbone)
        self.backbone = net

    def forward(self, x):
        x = F.interpolate(x, size=(self.input_size, self.input_size), mode="bilinear", align_corners=False)
        x = x.repeat(1, 3, 1, 1)
        return self.backbone(self.norm(x))


def count_params(model):
    return sum(p.numel() for p in model.parameters())

## 6. Training with early stopping on validation macro-F1
**Accuracy vs macro-F1.** Accuracy is the share of correct predictions, so it is dominated by the big classes. A model that never predicts *disgust* loses only about 1.5% accuracy.

F1 is the harmonic mean of precision and recall for one class:
- **precision**: when it says *fear*, how often is it really fear?
- **recall**: of all fearful faces, how many did it find?

**Macro-F1** is the plain average of the 7 per-class F1 scores, so every emotion counts equally, *disgust* included. That matches what the app needs: a player that never detects sadness is broken, whatever its accuracy.

**Early stopping:** after each epoch we measure validation macro-F1 and keep the best checkpoint. Training stops when the score hasn't improved for `patience` epochs, before the model starts overfitting. The test set plays no part in this.

Optimiser: AdamW with cosine learning-rate decay. On GPU, mixed precision (float16 autocast) roughly halves training time. Evaluation always runs in float32.

In [ ]:
@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    outs, labels = [], []
    for x, y in loader:
        outs.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
        labels.append(y)
    return torch.cat(outs), torch.cat(labels)


def basic_metrics(logits, labels):
    preds, y = logits.argmax(1).numpy(), labels.numpy()
    return {"accuracy": float(accuracy_score(y, preds)),
            "macro_f1": float(f1_score(y, preds, average="macro", labels=list(range(len(CLASSES))), zero_division=0))}


def train_model(name, model, param_groups, cfg):
    model.to(DEVICE)
    train_loader = make_loader(X_train, y_train, train=True)  # fresh seeded generator per model
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))
    optimizer = torch.optim.AdamW(param_groups, weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["max_epochs"])
    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_f1, best_epoch, bad_epochs, history = -1.0, 0, 0, []
    ckpt = OUT / f"{name}_best.pt"
    t0 = time.time()
    for epoch in range(1, cfg["max_epochs"] + 1):
        model.train()
        running, seen = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * len(y)
            seen += len(y)
        lr = optimizer.param_groups[-1]["lr"]
        scheduler.step()

        val_logits, val_labels = predict_logits(model, val_loader)
        val_loss = F.cross_entropy(val_logits, val_labels, weight=CLASS_WEIGHTS).item()
        m = basic_metrics(val_logits, val_labels)
        history.append({"epoch": epoch, "train_loss": running / seen, "val_loss": val_loss,
                        "val_accuracy": m["accuracy"], "val_macro_f1": m["macro_f1"], "lr": lr})
        improved = m["macro_f1"] > best_f1
        print(f"[{name}] epoch {epoch:>2}  train_loss {running / seen:.4f}  val_loss {val_loss:.4f}  "
              f"val_acc {m['accuracy']:.4f}  val_macroF1 {m['macro_f1']:.4f}{'  *' if improved else ''}")
        if improved:
            best_f1, best_epoch, bad_epochs = m["macro_f1"], epoch, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad_epochs += 1
            if bad_epochs >= cfg["patience"]:
                print(f"[{name}] early stop: no val macro-F1 gain for {cfg['patience']} epochs")
                break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return {"history": history, "best_epoch": best_epoch, "best_val_macro_f1": best_f1,
            "epochs_trained": epoch, "train_time_s": round(time.time() - t0, 1)}

### (a) Small CNN from scratch

In [ ]:
seed_everything()
cnn = SmallCNN(len(CLASSES), DATA_MEAN, DATA_STD)
print(f"SmallCNN parameters: {count_params(cnn):,}")
cnn_run = train_model("small_cnn", cnn, [{"params": cnn.parameters(), "lr": CNN_CFG["lr"]}], CNN_CFG)
print({k: v for k, v in cnn_run.items() if k != "history"})

### (b) Transfer learning: fine-tuning a pretrained backbone

In [ ]:
seed_everything()
tnet = TransferNet(TRANSFER_BACKBONE, len(CLASSES), TRANSFER_INPUT_SIZE, pretrained=not SMOKE_TEST)
head_params = [p for n, p in tnet.named_parameters() if n.startswith(tnet.head_prefix)]
body_params = [p for n, p in tnet.named_parameters() if not n.startswith(tnet.head_prefix)]
print(f"{TRANSFER_BACKBONE} parameters: {count_params(tnet):,} (head {sum(p.numel() for p in head_params):,})")
transfer_run = train_model("transfer", tnet, [
    {"params": body_params, "lr": TRANSFER_CFG["lr_backbone"]},
    {"params": head_params, "lr": TRANSFER_CFG["lr_head"]},
], TRANSFER_CFG)
print({k: v for k, v in transfer_run.items() if k != "history"})

In [ ]:
MODELS = {"small_cnn": cnn, "transfer": tnet}
RUNS = {"small_cnn": cnn_run, "transfer": transfer_run}
LABELS = {"small_cnn": "Small CNN (scratch)", "transfer": f"{TRANSFER_BACKBONE} (transfer)"}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for key, run in RUNS.items():
    ep = [h["epoch"] for h in run["history"]]
    c = MODEL_COLORS[key]
    axes[0].plot(ep, [h["train_loss"] for h in run["history"]], color=c, lw=2, ls="--", label=f"{LABELS[key]} train")
    axes[0].plot(ep, [h["val_loss"] for h in run["history"]], color=c, lw=2, label=f"{LABELS[key]} val")
    axes[1].plot(ep, [h["val_macro_f1"] for h in run["history"]], color=c, lw=2, label=LABELS[key])
    axes[1].plot(run["best_epoch"], run["best_val_macro_f1"], "o", ms=8, color=c, mec=SURFACE, mew=2)
axes[0].set(title="Weighted cross-entropy loss", xlabel="epoch")
axes[1].set(title="Validation macro-F1 (dot = kept checkpoint)", xlabel="epoch")
for ax in axes:
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Model selection (validation only)
The rule was fixed in the config before any test numbers existed: keep the smaller model unless the larger one beats it by more than `SELECTION_TOLERANCE` on **validation** macro-F1. Choosing by test score would make the test score optimistic, because it would no longer be measured on truly unseen data.

In [ ]:
cnn_val, tr_val = cnn_run["best_val_macro_f1"], transfer_run["best_val_macro_f1"]
smaller = min(MODELS, key=lambda k: count_params(MODELS[k]))
larger = max(MODELS, key=lambda k: count_params(MODELS[k]))
gain = RUNS[larger]["best_val_macro_f1"] - RUNS[smaller]["best_val_macro_f1"]
CHOSEN = larger if gain > SELECTION_TOLERANCE else smaller
selection = {
    "rule": f"prefer the smaller model unless the larger beats it by > {SELECTION_TOLERANCE} validation macro-F1",
    "val_macro_f1": {"small_cnn": cnn_val, "transfer": tr_val},
    "larger_minus_smaller": gain,
    "chosen": CHOSEN,
}
print(json.dumps(selection, indent=2))

## 8. Test-set evaluation
**Confusion matrix.** Rows are the true emotion, columns are the predicted emotion. The diagonal holds the correct predictions, and every off-diagonal cell is a specific mistake, such as "fear predicted as sad". The row-normalised version divides each row by its total, so the diagonal shows each class's recall. That makes the classes comparable despite the imbalance.

**Context:** published results on FER-2013 are typically 65–73% test accuracy, and human accuracy is estimated at about 65% (the dataset's labels are noisy). The numbers below are what this run measured.

In [ ]:
def test_report(model):
    logits, labels = predict_logits(model, test_loader)
    preds, y = logits.argmax(1).numpy(), labels.numpy()
    rep = classification_report(y, preds, labels=list(range(len(CLASSES))), target_names=CLASSES,
                                output_dict=True, zero_division=0)
    return {
        "accuracy": float(accuracy_score(y, preds)),
        "macro_f1": float(rep["macro avg"]["f1-score"]),
        "weighted_f1": float(rep["weighted avg"]["f1-score"]),
        "per_class": {c: {"precision": float(rep[c]["precision"]), "recall": float(rep[c]["recall"]),
                          "f1": float(rep[c]["f1-score"]), "support": int(rep[c]["support"])} for c in CLASSES},
        "confusion_matrix": confusion_matrix(y, preds, labels=list(range(len(CLASSES)))).tolist(),
    }, logits


def plot_confusion(cm, title, path):
    cm = np.array(cm)
    norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8))
    for ax, mat, fmt, sub in ((axes[0], cm, "d", "counts"), (axes[1], norm, ".2f", "row-normalised (recall)")):
        ax.imshow(norm, cmap=SEQ_BLUE, vmin=0, vmax=1)  # colour always encodes the row share
        for i in range(len(CLASSES)):
            for j in range(len(CLASSES)):
                ax.text(j, i, format(mat[i, j], fmt), ha="center", va="center", fontsize=8,
                        color="#ffffff" if norm[i, j] > 0.5 else INK)
        ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right")
        ax.set_yticks(range(len(CLASSES)), CLASSES)
        ax.set(xlabel="predicted", ylabel="true", title=sub)
        for s in ax.spines.values():
            s.set_visible(False)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()


TEST, TEST_LOGITS = {}, {}
for key, model in MODELS.items():
    TEST[key], TEST_LOGITS[key] = test_report(model)
    t = TEST[key]
    print(f"{LABELS[key]:<28} accuracy {t['accuracy']:.4f}  macro-F1 {t['macro_f1']:.4f}  weighted-F1 {t['weighted_f1']:.4f}")
    plot_confusion(t["confusion_matrix"],
                   f"{LABELS[key]}: test accuracy {t['accuracy']:.1%}, macro-F1 {t['macro_f1']:.3f}",
                   FIG / f"confusion_matrix_{key}.png")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
w = 0.38
x = np.arange(len(CLASSES))
for k, (key, off) in enumerate((("small_cnn", -w / 2), ("transfer", w / 2))):
    vals = [TEST[key]["per_class"][c]["f1"] for c in CLASSES]
    bars = ax.bar(x + off, vals, width=w - 0.02, color=MODEL_COLORS[key], label=LABELS[key])
    ax.bar_label(bars, fmt="%.2f", fontsize=7, color=INK_2, padding=2)
ax.set_xticks(x, CLASSES)
ax.set(ylim=(0, 1), ylabel="F1", title="Per-class test F1")
ax.grid(axis="y", color=GRID, lw=0.8)
ax.set_axisbelow(True)
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
fig.savefig(FIG / "per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Size and CPU inference latency
Latency is measured for a single image (batch 1), the way the app uses the model, on **one CPU thread**. That is closest to a browser running WebAssembly. We report the median and 90th percentile over 200 runs after a warm-up. The Colab CPU is not a laptop CPU, so the in-browser latency is measured again in Phase 3.

In [ ]:
import onnx
import onnxruntime as ort


def export_onnx(model, path):
    model = copy.deepcopy(model).cpu().eval()
    dummy = torch.rand(1, 1, IMG_SIZE, IMG_SIZE)
    kwargs = dict(input_names=["input"], output_names=["logits"], opset_version=17)
    try:
        torch.onnx.export(model, (dummy,), str(path), dynamo=False, **kwargs)
    except Exception as e:  # newer PyTorch builds without the TorchScript exporter
        print("TorchScript exporter unavailable, using dynamo exporter:", type(e).__name__, e)
        kwargs["opset_version"] = 18  # its native opset; down-converting to 17 is unreliable
        torch.onnx.export(model, (dummy,), str(path), dynamo=True, external_data=False, **kwargs)
    onnx.checker.check_model(onnx.load(str(path)))


def latency_ms(fn, runs=200, warmup=20):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return {"median": float(np.median(times)), "p90": float(np.percentile(times, 90))}


def ort_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 1
    so.inter_op_num_threads = 1
    return ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])


STATS = {}
prev_threads = torch.get_num_threads()
torch.set_num_threads(1)
for key, model in MODELS.items():
    onnx_path = ONNX_DIR / f"{key}.onnx"
    export_onnx(model, onnx_path)
    cpu_model = copy.deepcopy(model).cpu().eval()
    x = torch.rand(1, 1, IMG_SIZE, IMG_SIZE)
    with torch.inference_mode():
        torch_lat = latency_ms(lambda: cpu_model(x))
    sess = ort_session(onnx_path)
    xn = x.numpy()
    onnx_lat = latency_ms(lambda: sess.run(None, {"input": xn}))
    STATS[key] = {
        "params": count_params(model),
        "pytorch_checkpoint_bytes": (OUT / f"{key}_best.pt").stat().st_size,
        "onnx_bytes": onnx_path.stat().st_size,
        "cpu_latency_ms_pytorch_1thread": torch_lat,
        "cpu_latency_ms_onnxruntime_1thread": onnx_lat,
    }
    print(f"{LABELS[key]:<28} params {STATS[key]['params']:>11,}  onnx {STATS[key]['onnx_bytes'] / 1e6:6.2f} MB  "
          f"torch {torch_lat['median']:6.2f} ms  onnxruntime {onnx_lat['median']:6.2f} ms (median, 1 thread)")
torch.set_num_threads(prev_threads)

## 10. ONNX export and validation
**ONNX** (Open Neural Network Exchange) is a framework-neutral file format for a trained model: the computation graph plus its weights. PyTorch writes it, and **ONNX Runtime** runs it anywhere, including the browser through `onnxruntime-web` (WebAssembly/WebGPU). The browser therefore needs neither Python nor PyTorch.

Export can silently change behaviour, for example with an unsupported op or a different resize rule. To check it, we run **the entire test set** through both PyTorch and ONNX Runtime, then compare the logits, the probabilities and the predicted labels.

In [ ]:
chosen_cpu = copy.deepcopy(MODELS[CHOSEN]).cpu().eval()
sess = ort_session(ONNX_DIR / f"{CHOSEN}.onnx")
max_logit_diff = max_prob_diff = 0.0
agree = 0
onnx_preds = []
with torch.inference_mode():
    for i in range(len(X_test)):
        x = torch.from_numpy(X_test[i:i + 1]).unsqueeze(1).float() / 255.0
        pt = chosen_cpu(x).numpy()
        ox = sess.run(None, {"input": x.numpy()})[0]
        max_logit_diff = max(max_logit_diff, float(np.abs(pt - ox).max()))
        p_pt, p_ox = torch.softmax(torch.from_numpy(pt), 1), torch.softmax(torch.from_numpy(ox), 1)
        max_prob_diff = max(max_prob_diff, float((p_pt - p_ox).abs().max()))
        agree += int(pt.argmax() == ox.argmax())
        onnx_preds.append(int(ox.argmax()))

onnx_validation = {
    "model": CHOSEN,
    "images_compared": len(X_test),
    "max_abs_logit_diff": max_logit_diff,
    "max_abs_prob_diff": max_prob_diff,
    "argmax_agreement": agree / len(X_test),
    "onnx_test_accuracy": float(accuracy_score(y_test, onnx_preds)),
    "pytorch_test_accuracy": TEST[CHOSEN]["accuracy"],
}
print(json.dumps(onnx_validation, indent=2))
assert max_logit_diff < 1e-3, "ONNX output diverges from PyTorch"

shutil.copy(ONNX_DIR / f"{CHOSEN}.onnx", ONNX_DIR / "emotion_model.onnx")
meta = {
    "labels": CLASSES,
    "model": CHOSEN,
    "input": {"name": "input", "shape": [1, 1, IMG_SIZE, IMG_SIZE], "dtype": "float32",
              "preprocessing": "face crop -> grayscale (0.299R + 0.587G + 0.114B) -> resize to 48x48 -> divide by 255. "
                               "Normalisation is inside the model."},
    "output": {"name": "logits", "postprocessing": "softmax"},
}
(ONNX_DIR / "emotion_meta.json").write_text(json.dumps(meta, indent=2))

## 11. Error analysis
The code below lists which emotions get confused most, shows real misclassified test images for the top confusions, and checks how accuracy changes with the model's confidence. That last check is used to pick the confidence threshold for the browser in Phase 3.

In [ ]:
cm = np.array(TEST[CHOSEN]["confusion_matrix"])
norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
pairs = sorted(((norm[i, j], CLASSES[i], CLASSES[j], int(cm[i, j]))
                for i in range(len(CLASSES)) for j in range(len(CLASSES)) if i != j), reverse=True)
top_confusions = [{"true": t, "predicted": p, "share_of_true_class": float(s), "count": n} for s, t, p, n in pairs[:6]]
print(f"Top confusions for {LABELS[CHOSEN]}:")
for c in top_confusions:
    print(f"  {c['true']:>9} -> {c['predicted']:<9} {c['share_of_true_class']:.1%} of {c['true']} ({c['count']} images)")

probs = torch.softmax(TEST_LOGITS[CHOSEN], 1)
conf, pred = probs.max(1)
pred, conf = pred.numpy(), conf.numpy()
fig, axes = plt.subplots(3, 8, figsize=(12, 5.2))
for r, c in enumerate(top_confusions[:3]):
    ti, pi = CLASSES.index(c["true"]), CLASSES.index(c["predicted"])
    idx = np.where((y_test == ti) & (pred == pi))[0]
    idx = idx[np.argsort(-conf[idx])][:8]  # most confident mistakes first
    for k in range(8):
        ax = axes[r, k]
        ax.axis("off")
        if k < len(idx):
            ax.imshow(X_test[idx[k]], cmap="gray", vmin=0, vmax=255)
            ax.set_title(f"{conf[idx[k]]:.0%}", fontsize=8, color=INK_2)
    axes[r, 0].text(-0.15, 0.5, f"{c['true']}\n→ {c['predicted']}", ha="right", va="center", fontsize=9,
                    transform=axes[r, 0].transAxes)
fig.suptitle(f"Most confident test mistakes, {LABELS[CHOSEN]} (true → predicted, model confidence)")
fig.savefig(FIG / "error_examples.png", dpi=150, bbox_inches="tight")
plt.show()

confidence_curve = []
for t in (0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    keep = conf >= t
    confidence_curve.append({"threshold": t, "coverage": float(keep.mean()),
                             "accuracy": float((pred[keep] == y_test[keep]).mean()) if keep.any() else None})
print(f"\n{'threshold':>9} {'coverage':>9} {'accuracy':>9}")
for row in confidence_curve:
    acc = f"{row['accuracy']:.4f}" if row["accuracy"] is not None else "n/a"
    print(f"{row['threshold']:>9.1f} {row['coverage']:>9.1%} {acc:>9}")

## 12. Save results

In [ ]:
def hparams(key):
    base = {"batch_size": BATCH_SIZE, "optimizer": "AdamW", "scheduler": "cosine", "loss": "weighted cross-entropy",
            "augmentation": "RandomResizedCrop(0.8-1.0), HorizontalFlip, Rotation(10)", "amp": DEVICE.type == "cuda"}
    if key == "small_cnn":
        return {**base, **CNN_CFG}
    return {**base, **TRANSFER_CFG, "backbone": TRANSFER_BACKBONE, "pretrained": "ImageNet1K_V1",
            "input_upsampled_to": TRANSFER_INPUT_SIZE}


results = {
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "smoke_test": SMOKE_TEST,
    "seed": SEED,
    "dataset": {"name": "FER-2013", "source": "kaggle.com/datasets/msambare/fer2013", "classes": CLASSES,
                "split": f"official train/test; val = stratified {VAL_FRACTION:.0%} of train (seed {SEED})",
                "counts": split_counts, "class_weights": dict(zip(CLASSES, map(float, class_weights)))},
    "preprocessing": {"input_shape": [1, 1, IMG_SIZE, IMG_SIZE], "input_range": "[0, 1] (grayscale / 255)",
                      "normalisation": "inside model", "small_cnn_mean_std": [DATA_MEAN, DATA_STD]},
    "models": {key: {"label": LABELS[key], "hyperparameters": hparams(key),
                     "training": RUNS[key], "test": TEST[key], **STATS[key]} for key in MODELS},
    "selection": selection,
    "onnx_validation": onnx_validation,
    "error_analysis": {"model": CHOSEN, "top_confusions": top_confusions, "confidence_curve": confidence_curve},
    "environment": {"torch": torch.__version__, "torchvision": torchvision.__version__,
                    "onnx": onnx.__version__, "onnxruntime": ort.__version__, "python": platform.python_version(),
                    "device": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
                    "cpu": platform.processor() or platform.machine()},
}
(OUT / "results.json").write_text(json.dumps(results, indent=2))

print(f"\n{'model':<28}{'test acc':>10}{'macro-F1':>10}{'params':>12}{'ONNX MB':>9}{'ORT ms':>8}")
for key in MODELS:
    t, s = TEST[key], STATS[key]
    print(f"{LABELS[key]:<28}{t['accuracy']:>10.4f}{t['macro_f1']:>10.4f}{s['params']:>12,}"
          f"{s['onnx_bytes'] / 1e6:>9.2f}{s['cpu_latency_ms_onnxruntime_1thread']['median']:>8.2f}")
print("chosen:", CHOSEN)

In [ ]:
# Bundle everything except the .pt checkpoints (they stay in the Colab VM).
bundle = Path(OUT_DIR).parent / "moody_player_ml_outputs.zip"
import zipfile
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file() and f.suffix != ".pt":
            z.write(f, f.relative_to(OUT))
print("wrote", bundle, f"{bundle.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    pass

## 13. Results and discussion
*This section is written from `results.json` after the Colab run, so it only contains measured numbers.*